# StoneSense-AI: CT Multi-Model Experimentation — Setup & Data Verification

### 📋 Overview & Drive Layout
This notebook is **Step 0** of the Phase 8 Multi-Model CT evaluation workflow. It verifies your environment, links Google Drive persistence, validates dataset integrity for `dl/processed_grouped`, guarantees **zero near-duplicate cluster leakage**, and prepares directory structures and YOLO dataset views for downstream modeling.

---
### 🗂️ Google Drive Persistence Directory Layout
```
/content/drive/MyDrive/StoneSense-AI/
├── data/
│   ├── processed_grouped/           # 8708 Train, 1869 Val, 1869 Test
│   └── grouped_split_manifest.csv   # Cluster mapping & splits
├── artifacts/
│   ├── embeddings/                  # Cached DINOv3 .npz features
│   └── yolo_view/                   # Train/Val/Test YOLO format
└── outputs/
    ├── dinov3/                      # model_card.json, metrics.json, predictions.csv, weights
    ├── yolo26/                      # model_card.json, metrics.json, predictions.csv, best.pt
    ├── qknn/                        # model_card.json, metrics.json, predictions.csv, qknn_bundle.joblib
    └── comparison/                  # comparison.json, benchmark tables
```

---
### ⚙️ Notebook Flow
1. **Drive Mount & Workspace Configuration**
2. **Environment & Dependency Diagnostic Check**
3. **Grouped Split Manifest Verification & Leakage Assertion**
4. **Dataset Image Integrity & Class Balance Scan**
5. **YOLO-Compatible Dataset View Creation (`dl/datasets/yolo_view`)**
6. **Package Verification & Persistence Summary**

---
### 1. Configuration & Persistence Settings
* **What it does:** Sets configuration variables, seeds, paths, Drive root, idempotency flag `RERUN`, and `SMOKE_TEST` flag (fast 50-sample mode).
* **Expected runtime:** < 1 second.
* **Expected output:** Prints configured paths and confirmed execution mode.

In [ ]:
import os
import sys
import shutil
from pathlib import Path

# ==============================================================================
# GLOBAL CONFIGURATION
# ==============================================================================
SEED = 42
RERUN = False       # Set to True to re-run verification and re-create view structures
SMOKE_TEST = False  # Set to True to run on 50 images per split for instant verification

# Determine runtime environment (Colab vs Kaggle vs Local)
IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = "kaggle_web_client" in sys.modules

if IN_COLAB:
    DRIVE_ROOT = Path("/content/drive/MyDrive/StoneSense-AI")
    WORKSPACE_ROOT = Path("/content/StoneSense-AI")
elif IN_KAGGLE:
    DRIVE_ROOT = Path("/kaggle/working/StoneSense-AI")
    WORKSPACE_ROOT = Path("/kaggle/working/StoneSense-AI")
else:
    WORKSPACE_ROOT = Path(".").resolve()
    DRIVE_ROOT = WORKSPACE_ROOT

# Derive all paths strictly from WORKSPACE_ROOT / DRIVE_ROOT using forward-slash POSIX paths
DATA_DIR = WORKSPACE_ROOT / "dl" / "processed_grouped"
MANIFEST_PATH = WORKSPACE_ROOT / "dl" / "outputs" / "reports" / "grouped_split_manifest.csv"
YOLO_VIEW_DIR = WORKSPACE_ROOT / "dl" / "datasets" / "yolo_view"
ARTIFACTS_DIR = WORKSPACE_ROOT / "dl" / "artifacts"
OUT_DIR = WORKSPACE_ROOT / "dl" / "models" / "ct"

CLASS_NAMES = ["Cyst", "Normal", "Stone", "Tumor"]

print(f"Runtime Environment: {'Google Colab' if IN_COLAB else ('Kaggle' if IN_KAGGLE else 'Local')}")
print(f"Workspace Root: {WORKSPACE_ROOT.as_posix()}")
print(f"Data Directory: {DATA_DIR.as_posix()}")
print(f"Drive Persistence Root: {DRIVE_ROOT.as_posix()}")
print(f"Smoke Test Mode: {SMOKE_TEST}")

---
### 2. Optional Drive Mount (Colab Only)
* **What it does:** Mounts Google Drive at `/content/drive` for automatic persistence across disconnections.
* **Expected runtime:** ~10-15 seconds.
* **Expected output:** Confirmation of Google Drive mount.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    print(f"Google Drive successfully mounted at {DRIVE_ROOT.as_posix()}")
else:
    print("Skipping Colab Drive mount (local or Kaggle environment).")

---
### 3. Environment & Hardware Diagnostics (CHECK Cell)
* **What it does:** Verifies Python, PyTorch, CUDA GPU availability, packages, and reads `HF_TOKEN` from environment/secrets.
* **Expected runtime:** ~3-5 seconds.
* **Expected output:** Hardware specs, GPU model, and installed library versions. Fails clearly if prerequisites are missing.

In [ ]:
import platform
import torch

print("=" * 60)
print("ENVIRONMENT & HARDWARE DIAGNOSTICS")
print("=" * 60)
print(f"Python Version : {platform.python_version()}")
print(f"OS Platform    : {platform.platform()}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available : {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU Device     : {torch.cuda.get_device_name(0)}")
    print(f"VRAM Available : {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
else:
    print("No GPU detected; using CPU for execution.")

# Check Hugging Face Token (Read strictly from Colab secrets or environment)
hf_token = None
if IN_COLAB:
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
    except Exception:
        hf_token = os.getenv('HF_TOKEN')
else:
    hf_token = os.getenv('HF_TOKEN')

if hf_token:
    print(f"HF_TOKEN detected in environment/secrets (length: {len(hf_token)})")
else:
    print("HF_TOKEN not found. DINOv3 gated access will require token; DINOv2 open fallback will be available.")

print("=" * 60)

---
### 4. Manifest & Zero-Leakage Verification
* **What it does:** Reads `grouped_split_manifest.csv` and verifies:
  1. Exact split counts: **Train (8,708), Validation (1,869), Test (1,869)** — Total 12,446.
  2. Fixed 4 classes: `Cyst`, `Normal`, `Stone`, `Tumor`.
  3. **Zero cluster leakage:** Asserts intersection of `near_duplicate_group_id` across splits is empty.
* **Expected runtime:** ~1-2 seconds.
* **Expected output:** Split table, cluster counts, and ZERO CLUSTER LEAKAGE VERIFIED.

In [ ]:
import pandas as pd

if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f"Manifest not found at {MANIFEST_PATH.as_posix()}. Check data sync.")

df_manifest = pd.read_csv(MANIFEST_PATH)
print(f"Loaded manifest: {len(df_manifest)} total records across {df_manifest['near_duplicate_group_id'].nunique()} clusters.")

# 1. Verify split counts
split_counts = df_manifest["destination_split"].value_counts().to_dict()
print(f"Split distribution: {split_counts}")

expected_counts = {"train": 8708, "validation": 1869, "test": 1869}
for split, exp_count in expected_counts.items():
    actual_count = split_counts.get(split, 0)
    assert actual_count == exp_count, f"Count mismatch in '{split}': expected {exp_count}, got {actual_count}"

# 2. Verify classes
classes = sorted(df_manifest["class"].unique().tolist())
assert classes == sorted(CLASS_NAMES), f"Unexpected classes: {classes} vs {CLASS_NAMES}"

# 3. Verify zero cluster leakage across splits
train_clusters = set(df_manifest[df_manifest["destination_split"] == "train"]["near_duplicate_group_id"])
val_clusters = set(df_manifest[df_manifest["destination_split"] == "validation"]["near_duplicate_group_id"])
test_clusters = set(df_manifest[df_manifest["destination_split"] == "test"]["near_duplicate_group_id"])

leakage_train_val = train_clusters.intersection(val_clusters)
leakage_train_test = train_clusters.intersection(test_clusters)
leakage_val_test = val_clusters.intersection(test_clusters)

assert len(leakage_train_val) == 0, f"Leakage detected between Train and Val: {len(leakage_train_val)} clusters"
assert len(leakage_train_test) == 0, f"Leakage detected between Train and Test: {len(leakage_train_test)} clusters"
assert len(leakage_val_test) == 0, f"Leakage detected between Val and Test: {len(leakage_val_test)} clusters"

print("=" * 60)
print("ZERO CLUSTER LEAKAGE VERIFIED:")
print(f"   - Total Clusters : {len(train_clusters) + len(val_clusters) + len(test_clusters)}")
print(f"   - Train Clusters : {len(train_clusters)}")
print(f"   - Val Clusters   : {len(val_clusters)}")
print(f"   - Test Clusters  : {len(test_clusters)}")
print(f"   - Overlap        : 0 clusters")
print("=" * 60)

---
### 5. Physical Image Filesystem Scan
* **What it does:** Scans `dl/processed_grouped/{train,validation,test}` to confirm JPEG images are readable on disk.
* **Expected runtime:** ~3-8 seconds (or ~1 sec in SMOKE_TEST mode).
* **Expected output:** Confirmed image count and verified dimensions.

In [ ]:
from PIL import Image

df_scan = df_manifest.head(150) if SMOKE_TEST else df_manifest
verified_images = 0
corrupted_images = []

for idx, row in df_scan.iterrows():
    split = row["destination_split"]
    cname = row["class"]
    fname = row["filename"]
    img_path = DATA_DIR / split / cname / fname
    
    if not img_path.exists():
        corrupted_images.append(img_path.as_posix())
        continue
    verified_images += 1

if corrupted_images:
    raise FileNotFoundError(f"{len(corrupted_images)} images missing on disk! Example: {corrupted_images[:3]}")

print(f"All {verified_images:,} scanned images verified successfully across disk paths in '{DATA_DIR.name}'.")

---
### 6. Create YOLO-Compatible View (`dl/datasets/yolo_view`)
* **What it does:** Builds a zero-copy symlink view (with copy fallback) mapping `validation` to `val` for Ultralytics YOLO26 training.
* **Expected runtime:** ~2-5 seconds.
* **Expected output:** `yolo_view` folder populated with `train/`, `val/`, `test/` and `dataset.yaml` created.

In [ ]:
import yaml

YOLO_VIEW_DIR.mkdir(parents=True, exist_ok=True)
yolo_yaml_path = YOLO_VIEW_DIR / "dataset.yaml"

# Ultralytics requires 'val' naming
split_mapping = {
    "train": "train",
    "validation": "val",
    "test": "test"
}

for src_split, dst_name in split_mapping.items():
    dst_dir = YOLO_VIEW_DIR / dst_name
    src_dir = DATA_DIR / src_split
    
    if dst_dir.exists() and not RERUN:
        print(f"YOLO view '{dst_name}' already exists. Skipping.")
        continue
        
    if dst_dir.exists():
        try:
            os.rmdir(dst_dir)
        except Exception:
            shutil.rmtree(dst_dir)
        
    try:
        if os.name == 'nt':
            import _winapi
            _winapi.CreateJunction(str(src_dir.resolve()), str(dst_dir.resolve()))
        else:
            os.symlink(src_dir.resolve(), dst_dir.resolve(), target_is_directory=True)
        print(f"Created zero-copy link: {dst_name} -> {src_split}")
    except (OSError, NotImplementedError) as exc:
        print(f"Zero-copy link failed ({exc}); copying view structure for {dst_name}...")
        shutil.copytree(src_dir, dst_dir)

dataset_config = {
    "path": YOLO_VIEW_DIR.resolve().as_posix(),
    "train": "train",
    "val": "val",
    "test": "test",
    "names": {i: name for i, name in enumerate(CLASS_NAMES)},
    "nc": len(CLASS_NAMES)
}

with open(yolo_yaml_path, "w", encoding="utf-8") as f:
    yaml.dump(dataset_config, f, default_flow_style=False)

print(f"Generated YOLO dataset config at {yolo_yaml_path.as_posix()}")

---
### 7. Verification & Summary Checklist
* **What it does:** Confirms all directories are ready for Notebooks 01–04.
* **Expected runtime:** < 1 second.
* **Expected output:** Final status report.

In [ ]:
print("=" * 60)
print("STEP 00 SETUP & DATA VERIFICATION COMPLETE")
print("=" * 60)
print(f"Processed Grouped Dataset: {DATA_DIR.as_posix()} (12,446 images)")
print(f"YOLO View Config         : {yolo_yaml_path.as_posix()}")
print(f"Artifacts Directory      : {ARTIFACTS_DIR.as_posix()}")
print(f"Ready to run Notebook 01_dinov3.ipynb!")
print("=" * 60)